# Step 1 — tags 파싱·정규화

`dataset/cleaned_USvideos.csv` 로드 → tags 정규화 → `step1_tags_parsed.csv` 저장 (Step 2+ 입력).

- `|` 분리, 양끝 따옴표 제거, 소문자화, 선두 `#` 제거, 빈 토큰 제거
- 결측(`[none]`/NaN/빈값) → 빈 리스트(태그 없음 그룹)
- 정합성: 파싱 태그 수 vs `tag_cnt`

> 각 Step 노트북은 self-contained. 로드/전처리 중복은 구조화 tradeoff.

In [1]:
import pandas as pd

In [2]:
CSV = '../../dataset/cleaned_USvideos.csv'
df = pd.read_csv(CSV)
df = (df.sort_values('views')
        .drop_duplicates('video_id', keep='last')
        .reset_index(drop=True))
print('shape:', df.shape)

shape: (6249, 26)


In [3]:
# tags 정규화: | 분리 -> 따옴표/공백 제거 -> 선두 # 제거 -> 소문자
QUOTE = chr(34)

def parse_tags(raw):
    if pd.isna(raw):
        return []
    out = []
    for t in str(raw).split('|'):
        t = t.strip().strip(QUOTE).strip().lstrip('#').strip().lower()
        if t and t != '[none]':
            out.append(t)
    return out

df['tag_list'] = df['tags'].apply(parse_tags)
df['n_tags'] = df['tag_list'].apply(len)

In [4]:
# 정합성: 파싱 태그 수 vs 기존 tag_cnt
same = (df['n_tags'] == df['tag_cnt']).mean()
print('n_tags == tag_cnt 비율: {:.3f}'.format(same))
print('(n_tags - tag_cnt) 요약:')
print((df['n_tags'] - df['tag_cnt']).describe())
print()

no_tags = int((df['n_tags'] == 0).sum())
mean_t = df['n_tags'].mean()
med_t = df['n_tags'].median()
print('태그 없음 영상:', no_tags, '({:.1f}%)'.format(no_tags / len(df) * 100))
print('평균 태그 수: {:.2f} / 중앙값: {:.0f}'.format(mean_t, med_t))
print()

print('샘플 5개:')
for _, r in df.sample(5, random_state=0).iterrows():
    print('  [' + r['category'] + ']', r['tag_list'][:6])

n_tags == tag_cnt 비율: 1.000
(n_tags - tag_cnt) 요약:
count    6249.000000
mean       -0.000320
std         0.017889
min        -1.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         0.000000
dtype: float64

태그 없음 영상: 244 (3.9%)
평균 태그 수: 20.08 / 중앙값: 20

샘플 5개:
  [Autos & Vehicles] ['11foot8', 'low clearance crash', 'truck crash', 'train trestle', 'durham']
  [Sports] ['wwe', 'world wrestling entertainment', 'wrestling', 'wrestler', 'wrestle', 'superstars']
  [Entertainment] ['curly mozzarella sticks', 'curly fries', 'diy curly fries', 'curly fry recipe', 'curly fry', 'mozzarella sticks']
  [Music] ['this is not the end', 'quiet', 'womens march', 'milck']
  [Music] ['only you', 'parson james', 'pop', 'rca records label']


In [5]:
# Step 2+ 입력용 저장 (각 Step self-contained)
out = df[['video_id', 'category', 'log_views', 'tag_list']].copy()
out['tags_joined'] = out['tag_list'].apply(lambda x: '|'.join(x))
out = out.drop(columns='tag_list')
out.to_csv('step1_tags_parsed.csv', index=False)
print('saved: step1_tags_parsed.csv', out.shape)

saved: step1_tags_parsed.csv (6249, 4)
